# 03 — Anomaly Detection (Isolation Forest + SHAP)

**Integrity / anomaly-screening layer of the Football Betting Integrity Monitor.**
This notebook is *modeling only*. Formal hypothesis tests (KS, Mann–Whitney U, ANOVA)
live in `04_hypothesis_tests`.

### What this notebook does
Runs the **H2-vs-H3 experiment**: two Isolation Forests on the same matches but two
different feature representations.

| Model | Feature set | Scaling | Expectation |
|-------|-------------|---------|-------------|
| **Universal (H2)** | `model_natural` + `model_both` (35) | natural units, pooled across all leagues | over-flags mid-tier (T1/G1) — their wide-but-*normal* spreads sit in the pooled tail |
| **Tier-aware (H3)** | `model_z` + `model_both` (35) | per-league-season z-scored | balanced flagging — each league centred on its own normal |

The two representations **must stay distinct**. The contrast is driven by the z-set's
*per-league-season recentering*, which is **non-monotone across the pooled column**
(a wide-but-normal T1 spread and a tight-but-normal E0 spread can swap order in z-space).
A plain global rescale of the natural set would *not* reproduce this — Isolation Forest is
invariant to any monotone per-feature transform — so if both models ran on z-scored
features, H2's over-flagging failure mode would vanish and the experiment would collapse.

### Reading the results — non-negotiable framing
- **No ground-truth fixing labels exist.** This is unsupervised candidate flagging.
  Report **differential flagging rate by tier**, *never* "false positive rate."
- `home_win` is a result label, not an integrity signal — it never enters the IF
  (reserved for an optional supervised RF leg only).
- `contamination` is just the **global flag-rate knob** (fixed at 0.05), not a calibrated
  prior on fixing. Tiers are read *against* that fixed global rate.

## 1 · Load inputs and rebuild the two feature sets

Inputs from `02_features` (in `data/processed/`): `features.parquet` (8,915 rows),
`feature_roles.json`, `feature_dictionary.csv`.

Feature sets are rebuilt **from the `roles` dict** (single source of truth) rather than
hardcoded, then reconciled against the precomputed `universal_set_H2` / `tier_aware_set_H3`
lists baked in 02 — list equality (order included), so any drift between this notebook and
the feature stage fails loudly here.

Guards asserted: 35/35 features, 30 natural / 30 z / 5 shared booleans, `home_win` absent
from both sets, and the redundant `xmkt_div_abs(_z)` pair excluded.

In [3]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest

def find_repo_root(marker="data/processed"):
    p = Path.cwd()
    for cand in [p, *p.parents]:
        if (cand / marker).is_dir():
            return cand
    raise FileNotFoundError(f"couldn't locate '{marker}' from {Path.cwd()} upward")

REPO = find_repo_root()
PROC = REPO / "data" / "processed"

df    = pd.read_parquet(PROC / "features.parquet")
meta  = json.load(open(PROC / "feature_roles.json"))
roles = meta["roles"]

assert len(df) == 8915, f"expected 8915 rows, got {len(df)}"

def cols_for(*role_names):
    return [c for c, r in roles.items() if r in role_names]

universal    = cols_for("model_natural", "model_both")   # H2: pooled natural-unit
tier_aware   = cols_for("model_z", "model_both")          # H3: per-league-season z
model_both   = cols_for("model_both")
natural_cols = cols_for("model_natural")
z_cols       = cols_for("model_z")

# reconcile derived sets with the precomputed lists baked in 02 (order included)
assert universal  == meta["universal_set_H2"],  "universal set drift vs 02"
assert tier_aware == meta["tier_aware_set_H3"], "tier-aware set drift vs 02"
assert len(universal) == 35 and len(tier_aware) == 35
assert len(natural_cols) == 30 and len(z_cols) == 30 and len(model_both) == 5
assert "home_win" not in universal + tier_aware                          # label never in IF
assert not ({"xmkt_div_abs", "xmkt_div_abs_z"} & set(universal + tier_aware))  # excluded stay out
print(f"universal H2: {len(universal)} | tier-aware H3: {len(tier_aware)} | shared booleans: {len(model_both)}")

universal H2: 35 | tier-aware H3: 35 | shared booleans: 5


## 2 · Pre-imputation audit + `pinnacle_missing` verification

Imputation has **not** run yet — this audit snapshots the NaN mask *before* anything is
touched, so the numbers reflect the real coverage gaps.

What we check:
- **Natural-unit cols with NaN.** Expect the Pinnacle-derived columns (`pinnacle_drift_*`,
  `b365_vs_ps_close_*`) plus a few leading-row NaN in `roll_draw_rate` / `roll_drift_mag`
  (the `min_periods=10` season-boundary from 02).
- **z-set is already clean** (the z-guard zeroed NaN in 02) — asserted to be exactly 0.
- **`pinnacle_missing` alignment.** A *soft* check: agreement < 1.00 on any Pinnacle column
  surfaces a genuine discrepancy (a Pinnacle col NaN where the flag says present) rather
  than silently asserting it away.
- **Structural 25/26 gap.** football-data upload lag (cuts off ~Jan 2026) + thin early
  mid-tier coverage → ~50% Pinnacle missing in 25/26, heaviest in T1/G1. Shown as a
  league × season pivot of mean `pinnacle_missing`.

In [5]:
na_before = df[natural_cols].isna()
na_counts = na_before.sum()
print("=== natural-unit cols with NaN (pre-imputation) ===")
print(na_counts[na_counts > 0].sort_values(ascending=False)
      .to_frame("n_missing").assign(pct=lambda d: (d.n_missing / len(df) * 100).round(1)))

z_na = df[z_cols].isna().sum().sum()
print(f"\nz-set total NaN (expect 0, z-guard zeroed in 02): {z_na}")
assert z_na == 0, "z-set has NaN — z-guard did not run as expected"

pinn_cols = [c for c in natural_cols if c.startswith("pinnacle_") or c.startswith("b365_vs_ps_close")]
flag = df["pinnacle_missing"].astype(bool)
print("\n=== pinnacle_missing vs actual NaN in Pinnacle-derived cols (soft check) ===")
for c in pinn_cols:
    agree = (flag == df[c].isna()).mean()
    nan_n = int(df[c].isna().sum())
    print(f"  {c:24s} agreement={agree:.3f}  (NaN={nan_n})")

print("\n=== Pinnacle gap by league-season (mean pinnacle_missing) ===")
print(df.assign(pm=flag).groupby(["season", "league"])["pm"].mean().unstack().round(2).fillna(""))

=== natural-unit cols with NaN (pre-imputation) ===
                       n_missing  pct
pinnacle_drift_h             728  8.2
pinnacle_drift_d             728  8.2
pinnacle_drift_a             728  8.2
b365_vs_ps_close_a           680  7.6
b365_vs_ps_close_d           680  7.6
b365_vs_ps_close_h           680  7.6
b365_drift_d                  27  0.3
overround_change              27  0.3
implied_prob_sum_open         27  0.3
b365_drift_h                  27  0.3
b365_drift_a                  27  0.3
spread_change_d                1  0.0
spread_change_a                1  0.0
max_opening_spread             1  0.0
max_spread_change              1  0.0
opening_spread_a               1  0.0
opening_spread_d               1  0.0
opening_spread_h               1  0.0
spread_change_h                1  0.0

z-set total NaN (expect 0, z-guard zeroed in 02): 0

=== pinnacle_missing vs actual NaN in Pinnacle-derived cols (soft check) ===
  pinnacle_drift_h         agreement=1.000  (NaN=728)
  p

## 3 · Imputation — per-league-season median (natural cols only)

Plan recorded in the feature dictionary:
- **Natural-unit columns:** per-league-season median.
- **Pinnacle-derived columns:** same median fill, but the structural 25/26 gap stays
  visible to the model via `pinnacle_missing`, so the IF doesn't over-trust imputed
  sharp-money signal.
- **z-set + the five booleans:** untouched (z-set already clean).

**Cascade fallback.** A per-league-season median is itself NaN if a whole `(league, season)`
group has no observations for a column (thin early mid-tier Pinnacle coverage can cause this),
so the fill falls back league-median → global-median. Post-condition asserted: zero NaN
remain across all natural columns.

No train/test split here — this is full-data unsupervised flagging, so a full-data median is
correct. (If the optional supervised RF leg later uses CV, imputation would move inside the
folds.)

**COVID note:** 19/20 was z-scored against its own season in 02 (`is_covid_season` carried as
a modeling lever). For this first universal baseline all 8,915 rows are kept in;
`is_covid_season` is an `id` column, so it is *not* a feature the IF sees regardless.
Drop-from-training is a later lever.

In [6]:
na_before_total = df[natural_cols].isna().sum().sum()
for c in natural_cols:
    med_ls = df.groupby(["league", "season"])[c].transform("median")  # primary: per-league-season
    med_l  = df.groupby("league")[c].transform("median")              # fallback 1: per-league
    med_g  = df[c].median()                                           # fallback 2: global
    df[c]  = df[c].fillna(med_ls).fillna(med_l).fillna(med_g)

na_after = df[natural_cols].isna().sum().sum()
print(f"natural-col NaN: {na_before_total} -> {na_after}")
assert na_after == 0, "imputation left NaN — an entire feature is globally NaN?"
print("imputation complete; pinnacle_missing preserved as imputation marker ✓")

natural-col NaN: 4367 -> 0
imputation complete; pinnacle_missing preserved as imputation marker ✓


## 4 · Universal Isolation Forest (H2) → flagging rate by tier

First leg of the A/B: pooled IF on natural-unit features.

**Hyperparameters pinned** for reproducibility: `n_estimators=200`, `max_samples=256`
(equals `'auto'` here since n > 256), `contamination=0.05`, `random_state=42`.

**Score convention:** `decision_function < 0` ⇒ flagged; more negative ⇒ more anomalous.
The continuous `if_u_score` is stored alongside the boolean flag so we're not locked to the
5% threshold (enables threshold sweeps and is the quantity TreeExplainer will attribute in
the SHAP step).

**What to read:** global flag rate ≈ `contamination` by construction (sanity check), then
**flag rate by tier**. H2 predicts **mid (T1/G1) > elite (D1/E0)** — their wide-but-normal
spreads sit in the pooled tail.

⚠️ **Confound to name explicitly:** `pinnacle_missing` is *in* the universal set, and ~half
of 25/26 mid-tier rows share it. If mid-tier 25/26 flagging spikes, SHAP must confirm whether
it's driven by genuine spread/drift width or by the `pinnacle_missing` cluster. That flag is
doing its intended job, but it is a confound when reading the tier gap.

In [7]:
X = df[universal].copy()
assert X.isna().sum().sum() == 0
assert X.select_dtypes(include="object").shape[1] == 0, "non-numeric col in X"

IF_KW = dict(
    n_estimators=200,     # explicit (sklearn default 100); 200 for score stability
    max_samples=256,      # == 'auto' here since n>256; pinned for reproducibility
    max_features=1.0,
    contamination=0.05,   # fixes GLOBAL flag rate at 5% -> read tiers against this
    bootstrap=False,
    random_state=42,
    n_jobs=-1,
)
iso = IsolationForest(**IF_KW).fit(X)
# convention: decision_function < 0 == flagged; more negative == more anomalous
df["if_u_score"] = iso.decision_function(X)   # continuous — keep for threshold sweeps
df["if_u_flag"]  = iso.predict(X) == -1

print(f"global flag rate (≈contamination): {df['if_u_flag'].mean():.3f}\n")

tier_tbl = (df.groupby("tier")
              .agg(n=("if_u_flag","size"), flagged=("if_u_flag","sum"))
              .assign(flag_rate=lambda d: (d.flagged / d.n).round(3))
              .sort_values("flag_rate", ascending=False))
league_tbl = (df.groupby(["tier","league"])
                .agg(n=("if_u_flag","size"), flagged=("if_u_flag","sum"))
                .assign(flag_rate=lambda d: (d.flagged / d.n).round(3))
                .sort_values("flag_rate", ascending=False))
print("=== UNIVERSAL model — flag rate by TIER (H2 over-flag check) ==="); print(tier_tbl)
print("\n=== by league ==="); print(league_tbl)

global flag rate (≈contamination): 0.050

=== UNIVERSAL model — flag rate by TIER (H2 over-flag check) ===
             n  flagged  flag_rate
tier                              
mid_tier  4113      224      0.054
elite     4802      222      0.046

=== by league ===
                    n  flagged  flag_rate
tier     league                          
mid_tier G1      1666      176      0.106
elite    D1      2142      104      0.049
         E0      2660      118      0.044
mid_tier T1      2447       48      0.020


In [ ]:
# Diagnostic

print("=== flag rate by league × pinnacle_missing ===")
print(df.groupby(["league", "pinnacle_missing"])["if_u_flag"]
        .agg(n="size", flagged="sum")
        .assign(flag_rate=lambda d: (d.flagged / d.n).round(3))
        .reset_index())

print("\n=== flag rate excluding 25/26 (structural-gap season) ===")
mask = df["season"] != "2526"
print(df[mask].groupby("tier")["if_u_flag"]
        .agg(n="size", flagged="sum")
        .assign(flag_rate=lambda d: (d.flagged / d.n).round(3)))

=== flag rate by league × pinnacle_missing ===
  league  pinnacle_missing     n  flagged  flag_rate
0     D1                 0  1985      102      0.051
1     D1                 1   157        2      0.013
2     E0                 0  2490      116      0.047
3     E0                 1   170        2      0.012
4     G1                 0  1447      154      0.106
5     G1                 1   219       22      0.100
6     T1                 0  2265       42      0.019
7     T1                 1   182        6      0.033

=== flag rate excluding 25/26 (structural-gap season) ===
             n  flagged  flag_rate
tier                              
elite     4802      222      0.046
mid_tier  4113      224      0.054


## 5 · Tier-aware Isolation Forest (H3) → flag rate by league

Second leg of the A/B: identical IF, identical hyperparameters, but on the
**per-league-season z-scored** feature set. No imputation — the z-set is already clean
(z-guard zeroed NaN in 02).

### What the universal model told us (Cell 4) — read league-level, not tier-level
The tier rollup was misleading: it averaged two mid-tier leagues sitting at opposite ends.
The real universal-model finding is **league-specific, not tier-wide**:

| league | universal flag rate | vs global 0.05 |
|--------|--------------------|----------------|
| G1 (Greece) | **0.106** | 2× — most-flagged |
| D1 / E0 (elite) | ~0.05 | unremarkable |
| T1 (Turkey) | **0.020** | least-flagged |

Greece-high cross-validates the EDA (widest Pinnacle away drift, draw underpricing); the
25/26 Pinnacle gap was ruled out as the cause (effect persists with 25/26 excluded and
holds within `pinnacle_missing=0` rows).

### Concrete prediction for this cell
Per-league-season z recentres each league on **its own** normal, so a wide-but-normal
Greek spread that sat in the *pooled* tail collapses back toward zero. Prediction:
**G1 falls from ~0.106 toward ~0.05.** If z-scoring flattens Greece, that is the cleanest
possible demonstration of the recentering mechanism — the universal model flagged Greece
*for being Greece*, and the tier-aware model corrects it. Turkey's low rate, if it's genuine
market behaviour rather than a scaling effect, should move less.

**H2 vs H3 is the contrast itself** — not which model is "better." Universal preserves the
pooled distribution (mid-tier in the tail); tier-aware asks "anomalous *relative to this
league's own baseline*." Report league-level; the tier rollup is shown last only for
continuity with H2.

In [9]:
Xz = df[tier_aware].copy()
assert Xz.isna().sum().sum() == 0, "z-set has NaN — should be clean from 02"
assert Xz.select_dtypes(include="object").shape[1] == 0, "non-numeric col in z-set"

# SAME hyperparameters as the universal leg — only the feature representation changes
iso_t = IsolationForest(**IF_KW).fit(Xz)
df["if_t_score"] = iso_t.decision_function(Xz)   # more negative = more anomalous
df["if_t_flag"]  = iso_t.predict(Xz) == -1

print(f"global flag rate (≈contamination): {df['if_t_flag'].mean():.3f}\n")

# --- LEAGUE-LEVEL comparison first (tier rollup averages over the G1/T1 split) ---
def league_rates(flagcol):
    return (df.groupby(["tier", "league"])[flagcol]
              .agg(n="size", flagged="sum")
              .assign(rate=lambda d: (d.flagged / d.n).round(3))["rate"])

cmp = (pd.concat([league_rates("if_u_flag").rename("universal_H2"),
                  league_rates("if_t_flag").rename("tier_aware_H3")], axis=1)
         .assign(delta=lambda d: (d.tier_aware_H3 - d.universal_H2).round(3))
         .sort_values("universal_H2", ascending=False))
print("=== UNIVERSAL vs TIER-AWARE — flag rate by league ===")
print(cmp)

# how much do the two models overlap on *which* matches they flag?
both = int((df["if_u_flag"] & df["if_t_flag"]).sum())
u_only = int((df["if_u_flag"] & ~df["if_t_flag"]).sum())
t_only = int((~df["if_u_flag"] & df["if_t_flag"]).sum())
print(f"\nflagged by BOTH: {both} | universal-only: {u_only} | tier-only: {t_only}")

# tier rollup last, for continuity with Cell 4 only
print("\n=== tier rollup (shown for continuity — read league-level above) ===")
print(df.groupby("tier")["if_t_flag"]
        .agg(n="size", flagged="sum")
        .assign(flag_rate=lambda d: (d.flagged / d.n).round(3)))

global flag rate (≈contamination): 0.050

=== UNIVERSAL vs TIER-AWARE — flag rate by league ===
                 universal_H2  tier_aware_H3  delta
tier     league                                    
mid_tier G1             0.106          0.055 -0.051
elite    D1             0.049          0.054  0.005
         E0             0.044          0.051  0.007
mid_tier T1             0.020          0.043  0.023

flagged by BOTH: 328 | universal-only: 118 | tier-only: 118

=== tier rollup (shown for continuity — read league-level above) ===
             n  flagged  flag_rate
tier                              
elite     4802      251      0.052
mid_tier  4113      195      0.047


## 6 · SHAP — what drives each model's flags

`TreeExplainer` on each fitted Isolation Forest, attributing the anomaly score
(`decision_function`) to features, then aggregated to signal families (from the feature
dictionary) because per-feature beeswarms over 35 features × 2 models are unreadable.

### Reading conventions (grounded, not assumed)
- **Sign:** we verify empirically that `sum(SHAP)` tracks `decision_function`
  (corr ≈ +1). Since lower decision score = more anomalous, **negative SHAP pushes a match
  toward being flagged.** Headline importance uses **mean|SHAP|** (direction-agnostic).
- **Family size:** `total` = sum of mean|SHAP| within a family, so families with more
  features score higher mechanically. The `n_feat` and `per_feat` columns normalise this —
  read both. A family can have high `total` just by being large (spread = 12 features) or
  genuinely high per-feature impact (reversal = 3 features).
- `check_additivity=False`: the additivity check is unreliable for Isolation Forest; this
  is expected, not a bug.

### What we're testing
1. **Does Greece's universal over-flagging load on spread/drift width?** That's the EDA
   story (widest Pinnacle away drift, draw underpricing). If G1 universal flags are
   spread/drift-dominated, the mechanism narrative is confirmed, not just inferred from rates.
2. **Do the two disagreement sets load on different families?** universal-only flags
   (anomalous in pooled space) vs tier-only flags (anomalous vs own-league baseline) —
   the representational choice should show up as a family-mix difference.
3. **Confound check:** does `coverage` (`pinnacle_missing`) drive the tier-only flags? If so,
   some of the "anomalies" are really the 25/26 gap, not market behaviour.

In [11]:
import shap

def shap_frame(iso, cols):
    sv = shap.TreeExplainer(iso).shap_values(df[cols], check_additivity=False)
    return pd.DataFrame(sv, columns=cols, index=df.index)

sv_u = shap_frame(iso, universal)       # universal model from Cell 4 (named `iso`)
sv_t = shap_frame(iso_t, tier_aware)    # tier-aware model from Cell 5

# --- ground the sign convention empirically (don't assume) ---
corr = np.corrcoef(sv_u.sum(axis=1), df["if_u_score"])[0, 1]
print(f"corr(sum SHAP, decision_function) = {corr:.3f}  "
      f"-> negative SHAP pushes toward anomaly\n")

def fam_table(sv, cols, mask=None):
    s = sv.loc[mask] if mask is not None else sv
    fam = pd.Series({c: fam_map.get(c, "??") for c in cols})
    g = s.abs().mean().groupby(fam)
    return (pd.DataFrame({"total": g.sum(), "n_feat": g.size(), "per_feat": g.mean()})
              .sort_values("total", ascending=False).round(4))


fdict = pd.read_csv(PROC / "feature_dictionary.csv")
fam_map = dict(zip(fdict["column"], fdict["family"]))  # fdict loaded in Cell 1

print("=== UNIVERSAL (H2) — global family importance ===")
print(fam_table(sv_u, universal))
print("\n=== TIER-AWARE (H3) — global family importance ===")
print(fam_table(sv_t, tier_aware))

# --- test 1: is GREECE's universal flagging spread/drift-driven? ---
g1_uflag = df["if_u_flag"] & (df["league"] == "G1")
rest_uflag = df["if_u_flag"] & (df["league"] != "G1")
print(f"\n=== universal flags: G1 ({int(g1_uflag.sum())}) vs rest ({int(rest_uflag.sum())}) ===")
print(pd.concat([fam_table(sv_u, universal, g1_uflag)["per_feat"].rename("G1"),
                 fam_table(sv_u, universal, rest_uflag)["per_feat"].rename("rest")],
                axis=1).fillna(0).round(4))

# --- test 2 + 3: disagreement sets, each explained by ITS OWN model ---
u_only = df["if_u_flag"] & ~df["if_t_flag"]
t_only = ~df["if_u_flag"] & df["if_t_flag"]
print(f"\n=== disagreement sets (u_only={int(u_only.sum())}, t_only={int(t_only.sum())}) — per_feat ===")
print(pd.concat([fam_table(sv_u, universal, u_only)["per_feat"].rename("universal_only"),
                 fam_table(sv_t, tier_aware, t_only)["per_feat"].rename("tier_only")],
                axis=1).fillna(0).round(4))

corr(sum SHAP, decision_function) = 0.995  -> negative SHAP pushes toward anomaly

=== UNIVERSAL (H2) — global family importance ===
                         total  n_feat  per_feat
spread                  0.9066      12    0.0755
reversal                0.5685       3    0.1895
drift                   0.4428       7    0.0633
clv_crossbook           0.4228       6    0.0705
implied_prob_imbalance  0.2681       3    0.0894
coverage                0.1888       1    0.1888
steam                   0.0805       1    0.0805
rolling_temporal        0.0692       1    0.0692
margin                  0.0321       1    0.0321

=== TIER-AWARE (H3) — global family importance ===
                         total  n_feat  per_feat
spread                  0.9256      12    0.0771
reversal                0.5943       3    0.1981
drift                   0.4486       7    0.0641
clv_crossbook           0.4353       6    0.0725
implied_prob_imbalance  0.1921       3    0.0640
coverage                0.1871 

## 7 · Why does Greece over-flag? (clean mechanism test)

Cell 6's G1-vs-rest table compared families *among already-flagged matches* — it described
the **character** of Greek anomalies, not **why Greece gets flagged more often**. This cell
answers the rate question directly.

Method: signed SHAP (not |SHAP|), universal model, **all rows** (not just flagged). Since
negative SHAP pushes toward anomaly (corr check, Cell 6), a family where G1's mean signed
SHAP is *more negative* than the rest is systematically dragging Greek matches toward the
flagged end. `gap_toward_anomaly = rest_mean − G1_mean`: **positive ⇒ that family pushes
Greece toward anomaly**, ranked descending. The gaps sum to G1's total decision-score
deficit, so this is a complete additive decomposition of the rate gap — no family is hidden.

In [13]:
# signed SHAP across ALL rows: which families drag Greece toward the anomalous end
g1 = df["league"] == "G1"
fam = pd.Series({c: fam_map.get(c, "??") for c in universal})

# per-row family-summed signed SHAP (transpose-groupby: pandas dropped axis=1 groupby)
signed_by_fam = sv_u.T.groupby(fam).sum().T

mech = pd.DataFrame({"G1_mean":   signed_by_fam[g1].mean(),
                     "rest_mean": signed_by_fam[~g1].mean()})
mech["gap_toward_anomaly"] = (mech["rest_mean"] - mech["G1_mean"])  # +ve => pushes G1 toward anomaly
mech = mech.round(4).sort_values("gap_toward_anomaly", ascending=False)

print("=== which families push GREECE toward anomaly (universal, all rows) ===")
print(mech)

# additive sanity: family gaps should sum to G1's mean decision-score deficit
dfval = df["if_u_score"].values
print(f"\nsum of gaps: {mech['gap_toward_anomaly'].sum():.4f}")
print(f"G1 mean score: {dfval[g1.values].mean():.4f}  |  rest: {dfval[~g1.values].mean():.4f}  "
      f"|  deficit: {dfval[g1.values].mean() - dfval[~g1.values].mean():.4f}")

=== which families push GREECE toward anomaly (universal, all rows) ===
                        G1_mean  rest_mean  gap_toward_anomaly
spread                  -0.4350     0.0554              0.4904
drift                   -0.2682     0.0678              0.3360
clv_crossbook           -0.1584     0.0200              0.1784
coverage                -0.0622     0.0117              0.0739
implied_prob_imbalance  -0.0580     0.0091              0.0671
steam                   -0.0537     0.0063              0.0600
reversal                -0.0459     0.0005              0.0464
margin                  -0.0083    -0.0047              0.0036
rolling_temporal         0.0030    -0.0010             -0.0040

sum of gaps: 1.2518
G1 mean score: 0.1065  |  rest: 0.1416  |  deficit: -0.0350


## 8 · Persist scored matches (frozen artifact for 04 + Tableau)

Writes one parquet that downstream work pulls from, so `04_hypothesis_tests` and the Tableau
dashboard never re-run the Isolation Forests (no drift from the models validated here).

Contents: id columns + both flags + both decision scores + `pinnacle_missing` (confound
filter) + `flag_agreement` (both / universal_only / tier_only / neither) + a
dominant-family label per model.

**`dom_family_*` is a convenience filter only — not an analytical claim.** It's the family
with the most-negative *signed* SHAP for that row (pushed it most toward anomaly). Per Cell 7,
a single top-family-per-row collapses the signed-vs-absolute distinction, so it's fine for
slicing the dashboard ("show me spread-driven flags") but must not be read as *the* reason a
match flagged.

In [14]:
id_cols = [c for c, r in roles.items() if r == "id"]
missing_id = [c for c in id_cols if c not in df.columns]
if missing_id:
    print("WARN — id cols in roles but not in parquet (skipped):", missing_id)
id_cols = [c for c in id_cols if c in df.columns]

def dominant_family(sv, cols):
    fam = pd.Series({c: fam_map.get(c, "??") for c in cols})
    sbf = sv.T.groupby(fam).sum().T          # per-row family-summed signed SHAP
    return sbf.idxmin(axis=1)                # most-negative family = pushed most toward anomaly

scored = df[id_cols].copy()
scored["if_u_score"] = df["if_u_score"];  scored["if_u_flag"] = df["if_u_flag"]
scored["if_t_score"] = df["if_t_score"];  scored["if_t_flag"] = df["if_t_flag"]
scored["pinnacle_missing"] = df["pinnacle_missing"]
scored["dom_family_u"] = dominant_family(sv_u, universal)   # convenience filter — see markdown
scored["dom_family_t"] = dominant_family(sv_t, tier_aware)
scored["flag_agreement"] = np.select(
    [df["if_u_flag"] & df["if_t_flag"],
     df["if_u_flag"] & ~df["if_t_flag"],
     ~df["if_u_flag"] & df["if_t_flag"]],
    ["both", "universal_only", "tier_only"], default="neither")

out = PROC / "scored_matches.parquet"
scored.to_parquet(out, index=False)

rt = pd.read_parquet(out)                    # round-trip verification
assert rt.shape == scored.shape and list(rt.columns) == list(scored.columns)
assert rt["if_u_flag"].sum() == scored["if_u_flag"].sum(), "flag count changed on round-trip"
print(f"written: {out} | shape: {rt.shape}")
print("\nflag_agreement:\n", rt["flag_agreement"].value_counts())
print("\ndom_family_u on universal-flagged rows:\n", rt.loc[rt["if_u_flag"], "dom_family_u"].value_counts())

written: /Users/vladislavkivaev/Desktop/Final Project/football-betting-integrity-monitor/data/processed/scored_matches.parquet | shape: (8915, 18)

flag_agreement:
 flag_agreement
neither           8351
both               328
universal_only     118
tier_only          118
Name: count, dtype: int64

dom_family_u on universal-flagged rows:
 dom_family_u
spread           428
clv_crossbook      7
reversal           6
drift              5
Name: count, dtype: int64


### Conclusion — why Greece over-flags

Across all rows, **spread (0.49) and drift (0.34) account for ~two-thirds of Greece's total
push toward anomaly** (1.25), with every family except rolling_temporal pulling G1 more
negative than the rest. The rate-level finding (G1 flagged at 2× the global rate) now has a
feature-level cause that **independently cross-validates the EDA** — widest Pinnacle away
drift (71% wider than EPL), wide spreads — through a second method.

**Methodological note (important):** Cell 6 (mean|SHAP| *among flagged matches*) and Cell 7
(signed SHAP *across all rows*) answer different questions. Cell 6 ranked steam/implied-prob
high — that describes the *character* of Greek anomalies. Cell 7 ranks spread/drift high —
that explains *why the rate is elevated*, because it is measured over the population that
produces the rate rather than conditioned on the flag outcome. Cell 7 is the correct basis
for the mechanism claim; reporting Cell 6 alone would have given the wrong driver.

**Caveat:** the family gaps (Σ = 1.25) do **not** equal the raw decision-score deficit
(−0.035) — `shap_values` on Isolation Forest is in raw-score space (pre-`offset_`) and
`check_additivity=False` waives the sum-to-output guarantee, so this is a scale/offset
artifact, not a leak. The result rests on the *ranking and sign* of contributions
(every family except rolling_temporal pushes G1 toward anomaly, spread/drift dominant),
which are robust — not on an exact additive identity.

## Modeling findings — H2 & H3

**Setup.** Two Isolation Forests on the same 8,915 matches, differing only in feature
representation: universal (natural-unit, pooled) vs tier-aware (per-league-season z-scored).
`contamination = 0.05` fixes the global flag rate; results read as **differential flagging
rate by tier/league**, never "false positive rate" (no ground-truth fixing labels exist).

**H2 — "mid-tier less efficiently priced than elite": rejected as a tier claim.**
The universal model does not flag the mid-tier as a block. By league: G1 0.106, D1 0.049,
E0 0.044, T1 0.020 — Greece is the most-flagged and Turkey the *least*-flagged league in the
dataset, sitting at opposite ends. The tier average (mid 0.054 vs elite 0.046) is a
misleading midpoint of that split. Ruled out the 25/26 Pinnacle gap as a cause (effect
persists with 25/26 excluded and within `pinnacle_missing = 0` rows).
*Defensible reframe:* pooled anomaly flagging concentrates in **Greece specifically**,
driven by spread and drift width (SHAP, Cell 7) — a league-level, not tier-level, effect.

**H3 — "anomalies identifiable from odds features, tier-aware calibration rebalances":
supported.** Per-league-season z-scoring pulls Greece from 0.106 → 0.055 (Δ −0.051),
nudges Turkey 0.020 → 0.043, and leaves the elites near 0.05 — every league converges on
baseline. The two models flag substantively *different matches*: 328 shared, but 118
universal-only and 118 tier-only. SHAP on the disagreement sets shows the mechanism —
universal-only flags load on **magnitude families** (spread, steam), tier-only flags load on
the **scale-free booleans** (reversal, coverage), because z-scoring compresses magnitudes and
the booleans (`model_both`, never z-scored) carry relatively more weight in z-space.

**Confound logged.** `coverage` (`pinnacle_missing`) is a non-trivial SHAP contributor to
G1 and tier-only flags. It does not drive the *rate* difference (cleared in Cell 4), but a
minority of individual flags lean partly on a data-coverage signal and should be reported or
down-weighted, not presented as pure market anomalies.

**Frozen artifact.** `data/processed/scored_matches.parquet` — both flags, both scores,
`flag_agreement`, `pinnacle_missing`, dominant-family labels. Source of truth for
`04_hypothesis_tests` and Tableau.

In [15]:
import json
scored = pd.read_parquet(PROC / "scored_matches.parquet")   # fresh reload from disk

print("=== structural ===")
print("shape:", scored.shape, "| total nulls (expect 0):", int(scored.isna().sum().sum()))
assert scored["if_u_flag"].dtype == bool and scored["if_t_flag"].dtype == bool, "flags not bool after round-trip"

print("\n=== internal consistency (no refit needed) ===")
# sklearn identity: predict == -1  <=>  decision_function < 0
assert (scored["if_u_flag"] == (scored["if_u_score"] < 0)).all(), "universal flag/score mismatch"
assert (scored["if_t_flag"] == (scored["if_t_score"] < 0)).all(), "tier-aware flag/score mismatch"
recon = np.select(
    [scored.if_u_flag & scored.if_t_flag, scored.if_u_flag & ~scored.if_t_flag, ~scored.if_u_flag & scored.if_t_flag],
    ["both", "universal_only", "tier_only"], default="neither")
assert (recon == scored["flag_agreement"]).all(), "flag_agreement not reconstructable from flags"
print("flag==score<0 ✓   flag_agreement self-consistent ✓")

print("\n=== rate reconciliation (must match Cells 4/5) ===")
print("global  u:", round(scored.if_u_flag.mean(), 3), "| t:", round(scored.if_t_flag.mean(), 3))
print(scored.groupby("league")[["if_u_flag", "if_t_flag"]].mean().round(3))

print("\n=== key uniqueness ===")
key = [c for c in ["match_date", "home_team", "away_team"] if c in scored.columns]
dups = int(scored.duplicated(key).sum())
print(f"key {key}: {dups} duplicates", "✓" if dups == 0 else "← investigate")

=== structural ===
shape: (8915, 18) | total nulls (expect 0): 0

=== internal consistency (no refit needed) ===
flag==score<0 ✓   flag_agreement self-consistent ✓

=== rate reconciliation (must match Cells 4/5) ===
global  u: 0.05 | t: 0.05
        if_u_flag  if_t_flag
league                      
D1          0.049      0.054
E0          0.044      0.051
G1          0.106      0.055
T1          0.020      0.043

=== key uniqueness ===
key ['match_date', 'home_team', 'away_team']: 0 duplicates ✓
